# Lyrics Language Detection & Translation

## 1. Imports & Setup

In [ ]:
import fasttext
import pandas as pd
from time import sleep
from random import randint
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm

tqdm.pandas()

## 2. Basic Cleanup

In [2]:
input_file = 'data/results/tracks_data.csv'
tracks = pd.read_csv(input_file, encoding='utf-8')
lyrics = tracks[tracks['lyrics_text'].notna() & (tracks['lyrics_text'].str.strip() != '')]
lyrics.shape

(50172, 17)

In [3]:
cleaned = lyrics['lyrics_text'].apply(lambda x: re.sub("’", "'", re.sub(r" +", " " , re.sub(r"[\n\r\t]", " ", x))))
lyrics = lyrics.assign(lyrics_clean = cleaned)
lyrics.head(3)


,track_uri,music_title,music_artist,music_album,found_title_spotify,found_album_spotify,found_artist_spotify,found_genius_title,found_genius_artist,genius_track_url,lyrics_language,artist_genres,album_genres,analysis_url,lyrics_text,success,error_message,lyrics_clean
7,1gFNm7cXfG1vSMcxPpSxec,004_The Beatles - Ob-La-Di Ob-La-Da,<unknown>,cd 1,"Ob-La-Di, Ob-La-Da - Remastered 2009",The Beatles (Remastered),The Beatles,"Ob-La-Di, Ob-La-Da",The Beatles,https://genius.com/The-beatles-ob-la-di-ob-la-...,NaN,"['beatlesque', 'british invasion', 'classic ro...",[],https://api.spotify.com/v1/audio-analysis/1gFN...,Desmond has a barrow in the marketplace\nMolly...,True,NaN,Desmond has a barrow in the marketplace Molly ...
8,6j67aNAPeQ31uw4qw4rpLa,005_The Beatles - Wild Honey Pie,<unknown>,cd 1,Wild Honey Pie - Remastered 2009,The Beatles (Remastered),The Beatles,Wild Honey Pie,The Beatles,https://genius.com/The-beatles-wild-honey-pie-...,NaN,"['beatlesque', 'british invasion', 'classic ro...",[],https://api.spotify.com/v1/audio-analysis/6j67...,Honey pie\nHoney pie\n\n\nHoney pie\nHoney pie...,True,NaN,Honey pie Honey pie Honey pie Honey pie Honey ...
9,5Z3Rd1fMcaty8g5Pn7yhBQ,006_The Beatles - The Continuing Story of Bungal,<unknown>,cd 1,The Continuing Story Of Bungalow Bill - Remast...,The Beatles (Remastered),The Beatles,The Continuing Story of Bungalow Bill,The Beatles,https://genius.com/The-beatles-the-continuing-...,NaN,"['beatlesque', 'british invasion', 'classic ro...",[],https://api.spotify.com/v1/audio-analysis/5Z3R...,"Hey, Bungalow Bill\nWhat did you kill, Bungalo...",True,NaN,"Hey, Bungalow Bill What did you kill, Bungalow..."


## 3. Language Detection via FastText

In [4]:
# Load the Fasttext pretrained model
PRETRAINED_MODEL_PATH = 'data/lid.176.bin'
model = fasttext.load_model(PRETRAINED_MODEL_PATH)

In [5]:
# Predict language for each lyrics line
lyrics = lyrics.assign(lyrics_language = lyrics['lyrics_clean'].apply(lambda x: model.predict(x)[0][0]))
lyrics.head(3)

,track_uri,music_title,music_artist,music_album,found_title_spotify,found_album_spotify,found_artist_spotify,found_genius_title,found_genius_artist,genius_track_url,lyrics_language,artist_genres,album_genres,analysis_url,lyrics_text,success,error_message,lyrics_clean
7,1gFNm7cXfG1vSMcxPpSxec,004_The Beatles - Ob-La-Di Ob-La-Da,<unknown>,cd 1,"Ob-La-Di, Ob-La-Da - Remastered 2009",The Beatles (Remastered),The Beatles,"Ob-La-Di, Ob-La-Da",The Beatles,https://genius.com/The-beatles-ob-la-di-ob-la-...,__label__en,"['beatlesque', 'british invasion', 'classic ro...",[],https://api.spotify.com/v1/audio-analysis/1gFN...,Desmond has a barrow in the marketplace\nMolly...,True,NaN,Desmond has a barrow in the marketplace Molly ...
8,6j67aNAPeQ31uw4qw4rpLa,005_The Beatles - Wild Honey Pie,<unknown>,cd 1,Wild Honey Pie - Remastered 2009,The Beatles (Remastered),The Beatles,Wild Honey Pie,The Beatles,https://genius.com/The-beatles-wild-honey-pie-...,__label__en,"['beatlesque', 'british invasion', 'classic ro...",[],https://api.spotify.com/v1/audio-analysis/6j67...,Honey pie\nHoney pie\n\n\nHoney pie\nHoney pie...,True,NaN,Honey pie Honey pie Honey pie Honey pie Honey ...
9,5Z3Rd1fMcaty8g5Pn7yhBQ,006_The Beatles - The Continuing Story of Bungal,<unknown>,cd 1,The Continuing Story Of Bungalow Bill - Remast...,The Beatles (Remastered),The Beatles,The Continuing Story of Bungalow Bill,The Beatles,https://genius.com/The-beatles-the-continuing-...,__label__en,"['beatlesque', 'british invasion', 'classic ro...",[],https://api.spotify.com/v1/audio-analysis/5Z3R...,"Hey, Bungalow Bill\nWhat did you kill, Bungalo...",True,NaN,"Hey, Bungalow Bill What did you kill, Bungalow..."


## 4. English vs Non-English Split

In [6]:
# Retrieve songs where complete text is labelled as "english" 
eng_lyrics = lyrics.loc[lyrics['lyrics_language'] == '__label__en']
eng_lyrics.shape


(35478, 18)

In [7]:
# Retrieve non-English songs
oth_lyrics = lyrics.loc[lyrics['lyrics_language'] != '__label__en']
oth_lyrics.shape


(14694, 18)

## 5. Translate Non-English Lyrics

In [8]:
MAX_LEN = 5000  # Max characters for GoogleTranslator

LANG_MAP = {
    'als': 'de',
    'nn': 'no',
    'oc': 'fr',
    'cbk': 'es',
    'ia': 'la',
    'nds': 'de',
    'br': 'fr',
    'sh': 'sr',
    'pms': 'it',
    'bar': 'de',
}

def split_text(text, max_len=MAX_LEN):
    words = text.split()
    chunks = []
    current_chunk = []
    current_len = 0

    for word in words:
        if current_len + len(word) + 1 > max_len:
            chunks.append(" ".join(current_chunk))
            current_chunk = [word]
            current_len = len(word)
        else:
            current_chunk.append(word)
            current_len += len(word) + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

def translate_long_text(text, source_lang, target_lang='en'):
    chunks = split_text(text, MAX_LEN)
    translated_chunks = []
    for chunk in chunks:
        translated_chunks.append(
            GoogleTranslator(source=source_lang, target=target_lang).translate(chunk)
        )
    return " ".join(translated_chunks)

def detect_and_translate(row):
    lang = row['lyrics_language'].replace('__label__', '')
    lang = LANG_MAP.get(lang, lang)  # Map dialects to supported langs
    text = row['lyrics_clean']

    if lang != 'en':
        try:
            return translate_long_text(text, source_lang=lang)
        except Exception as e:
            print(f"Translation error for key {row.get('key', 'N/A')}: {e}")
            return text  # Fallback to original if translation fails
    else:
        return text

# Apply translation to all rows
lyrics['lyrics_en'] = lyrics.progress_apply(detect_and_translate, axis=1)

  0%|          | 0/50172 [00:00<?, ?it/s]

Translation error for key N/A: Request exception can happen due to an api connection error. Please check your connection and try again
Translation error for key N/A: Request exception can happen due to an api connection error. Please check your connection and try again
Translation error for key N/A: sequence item 0: expected str instance, NoneType found
Translation error for key N/A: sequence item 0: expected str instance, NoneType found
Translation error for key N/A: sequence item 0: expected str instance, NoneType found
Translation error for key N/A: sequence item 0: expected str instance, NoneType found
Translation error for key N/A: sequence item 0: expected str instance, NoneType found
Translation error for key N/A: he --> No support for the provided language.
Please select on of the supported languages:
{'afrikaans': 'af', 'albanian': 'sq', 'amharic': 'am', 'arabic': 'ar', 'armenian': 'hy', 'assamese': 'as', 'aymara': 'ay', 'azerbaijani': 'az', 'bambara': 'bm', 'basque': 'eu', 'b

In [9]:
output_file = 'data/results/tracks_data_translated.csv'
lyrics.to_csv(output_file, index=False, encoding='utf-8')

print(f"Saved translated dataset with new column 'lyrics_en' to {output_file}")

Saved translated dataset with new column 'lyrics_en' to data/results/tracks_data_translated.csv


In [12]:
# %% [markdown]
# ## 6. Translation Summary

# %%
import pandas as pd

# Load the translated dataset
translated_df = pd.read_csv('data/results/tracks_data_translated.csv', encoding='utf-8')

print("\n=== Translation Summary ===")

# Total rows
total_tracks = len(translated_df)
print(f"Total tracks processed: {total_tracks}")

# How many tracks were originally detected as English
num_en_original = (translated_df['lyrics_language'] == '__label__en').sum()
print(f"Originally English lyrics: {num_en_original}")

# Non-English tracks
num_non_en = total_tracks - num_en_original
print(f"Originally Non-English lyrics: {num_non_en}")

# Count by each detected language
print("\nBreakdown by detected language:")
lang_counts = translated_df['lyrics_language'].value_counts()
for lang, count in lang_counts.items():
    print(f"  {lang.replace('__label__','')}: {count}")

# Check how many have a non-empty translated version
num_translated = (
    (translated_df['lyrics_language'] != '__label__en') &
    translated_df['lyrics_en'].notna() &
    (translated_df['lyrics_en'].str.strip() != '')
).sum()
print(f"\nNon-English tracks that received a translation: {num_translated}")



=== Translation Summary ===
Total tracks processed: 50172
Originally English lyrics: 35478
Originally Non-English lyrics: 14694

Breakdown by detected language:
  en: 35478
  de: 10908
  es: 952
  it: 495
  fr: 477
  ko: 335
  ru: 328
  sv: 218
  pt: 148
  ja: 136
  tr: 90
  nl: 53
  pl: 45
  da: 39
  is: 35
  lt: 35
  fi: 33
  la: 30
  sr: 26
  no: 22
  id: 19
  tl: 18
  ro: 15
  als: 13
  uk: 12
  nn: 12
  he: 12
  el: 10
  bg: 9
  sl: 9
  zh: 9
  hi: 8
  hr: 8
  ar: 8
  cs: 7
  sq: 7
  nds: 6
  vi: 6
  fa: 6
  gl: 6
  hu: 6
  et: 5
  gd: 5
  ms: 4
  ga: 4
  eu: 4
  ku: 4
  sh: 3
  lmo: 3
  bs: 3
  ht: 3
  sw: 3
  pa: 3
  br: 3
  ne: 2
  nap: 2
  lv: 2
  tt: 2
  eo: 2
  oc: 2
  sk: 2
  bn: 2
  sa: 2
  jbo: 1
  su: 1
  be: 1
  mn: 1
  cbk: 1
  am: 1
  af: 1
  pms: 1
  uz: 1
  kk: 1
  hsb: 1
  ia: 1
  bar: 1
  ca: 1
  mg: 1
  as: 1
  hy: 1
  th: 1

Non-English tracks that received a translation: 14694


In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm.auto import tqdm
import os
import csv

tqdm.pandas()

MAX_LEN = 5000  # max chars per chunk for GoogleTranslator

LANG_MAP = {
    'als': 'de',
    'nn': 'no',
    'oc': 'fr',
    'cbk': 'es',
    'ia': 'la',
    'nds': 'de',
    'br': 'fr',
    'sh': 'sr',
    'pms': 'it',
    'bar': 'de',
}


def split_text(text, max_len=MAX_LEN):
    words = text.split()
    chunks = []
    current_chunk = []
    current_len = 0

    for word in words:
        if current_len + len(word) + 1 > max_len:
            chunks.append(" ".join(current_chunk))
            current_chunk = [word]
            current_len = len(word)
        else:
            current_chunk.append(word)
            current_len += len(word) + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

def translate_long_text(text, source_lang, target_lang='en'):
    chunks = split_text(text, MAX_LEN)
    translated_chunks = []
    for chunk in chunks:
        translated_chunk = GoogleTranslator(source=source_lang, target=target_lang).translate(chunk)
        translated_chunks.append(translated_chunk)
    return " ".join(translated_chunks)

def detect_and_translate(row):
    lang = row['lyrics_language'].replace('__label__', '')
    text = row['lyrics_clean']
    lang = LANG_MAP.get(lang, lang)

    if lang != 'en':
        try:
            return translate_long_text(text, source_lang=lang)
        except Exception as e:
            print(f"Translation error for key {row['key']}: {e}")
            return text
    else:
        return text

# Parameters
output_file = 'data/track_lyrics_translated.csv'
offset = 0  # set your offset here

# Open the output CSV appropriately
mode = 'w' if offset == 0 else 'a'
header = offset == 0

with open(output_file, mode, newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    if header:
        writer.writerow(['key', 'lyrics', 'lyrics_en'])

    for idx, row in tqdm(lyrics.iterrows(), total=len(lyrics)):
        if idx < offset:
            continue
        translated = detect_and_translate(row)
        writer.writerow([row['key'], row['lyrics'], translated])


  0%|          | 0/42075 [00:00<?, ?it/s]

Translation error for key lyrics:5sT164B7XL4S0h7zLuZyvi: sequence item 0: expected str instance, NoneType found
Translation error for key lyrics:4pq5Sqf6HUJe8B1iyYFh5i: sequence item 0: expected str instance, NoneType found
Translation error for key lyrics:49xGCjApgJYMQavlJg2X6V: sequence item 0: expected str instance, NoneType found
Translation error for key lyrics:2WjYE9tQDHuxe86hXD0cRs: sequence item 0: expected str instance, NoneType found
Translation error for key lyrics:3zglBTVFKirguQYcfcSh7o: zh --> No support for the provided language.
Please select on of the supported languages:
{'afrikaans': 'af', 'albanian': 'sq', 'amharic': 'am', 'arabic': 'ar', 'armenian': 'hy', 'assamese': 'as', 'aymara': 'ay', 'azerbaijani': 'az', 'bambara': 'bm', 'basque': 'eu', 'belarusian': 'be', 'bengali': 'bn', 'bhojpuri': 'bho', 'bosnian': 'bs', 'bulgarian': 'bg', 'catalan': 'ca', 'cebuano': 'ceb', 'chichewa': 'ny', 'chinese (simplified)': 'zh-CN', 'chinese (traditional)': 'zh-TW', 'corsican': 'co'

In [ ]:
# View example translations
lyrics.loc[lyrics['full_lyrics'] != '__label__en'][['lyrics', 'lyrics_en']].head(10)


## 6. Save Translated Output

In [ ]:
output_translated_file = 'data/results/track_lyrics_translated.csv'
lyrics[['key','lyrics', 'lyrics_en']].to_csv(output_translated_file, index=False, encoding='utf-8')